## This notebook contains script to analyze therapies patterns for five popular drugs

### Purpose
This notebook is designed to analyze and test the performance of the therapy detection algorithm by varying its key parameters. The main goal is to understand how changes in parameters—specifically `sd_f` and `sliding_window` - affect the final therapy-level statistics, particularly the error metrics (SMAE). The analysis will be conducted for five selected drugs with a high number of prescriptions to ensure a robust evaluation.

### Chosen drugs:
- simvastatin
- quetiapine
- amitriptyline
- warfarin
- bisoprolol

In [ ]:
drug_list = ['warfarin', 'quetiapine', 'amitriptyline', 'simvastatin', 'bisoprolol']

### Configuration and Hail tables loading

In [ ]:
import pyspark
import dxpy
import hail as hl
import random
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import sys
import multiprocessing as mp
import numpy as np
import random
import seaborn as sns
from tqdm import tqdm  
from matplotlib.colors import LogNorm
import itertools    

sys.path.append('../')
from functions.daily_dose_calc_utils import prepare_and_sort_data, calculate_intervals, calculate_daily_dose
from functions.therapy_detection_utils import merge_consecutive_records, process_therapies_with_dose_changes


In [ ]:
sc = pyspark.SparkContext()
spark = pyspark.sql.SparkSession(sc)
hl.init(sc=sc, default_reference='GRCh38')

In [ ]:
from datetime import datetime
print(f'Timestamp: {datetime.now()}')
print(f'Instance type: {dxpy.describe(dxpy.JOB_ID)["instanceType"]}')
print(f'Hail version: {hl.version()}')
print(f'Spark version: {spark.version}')

In [ ]:
input_database = 'prescriptions_db'
input_prescriptions_tb = 'filtered_prescriptions_with_doses_v5.0.0.ht'

In [ ]:
input_db_id = dxpy.find_one_data_object(name=input_database, classname='database', project=dxpy.PROJECT_CONTEXT_ID)['id']
input_prescriptions_ht = hl.read_table(f'dnax://{input_db_id}/{input_prescriptions_tb}')

In [ ]:
input_prescriptions_ht = input_prescriptions_ht.drop(
    'drug_name',
    'tokenized_drug_name',
    'matched_code',
    'match_mode'
)

### Filter Hail table for our drugs

In [ ]:
n_sample = 5000

In [ ]:
import os
os.makedirs('../data', exist_ok=True)

In [ ]:
for drug_name in drug_list[:1]:
    ht_filtered = input_prescriptions_ht.filter(input_prescriptions_ht.substance == drug_name)
    
    unique_eids_list = list(ht_filtered.aggregate(hl.agg.collect_as_set(ht_filtered.eid)))
    k = min(n_sample, len(unique_eids_list))
    sampled_eids = random.sample(unique_eids_list, k=k)
    sampled_eids_set = hl.literal(set(sampled_eids))
    
    ht_filtered = ht_filtered.filter(sampled_eids_set.contains(ht_filtered.eid))
    ht_filtered = ht_filtered.persist()
    
    ht_prepared = prepare_and_sort_data(ht_filtered)
    ht_with_intervals = calculate_intervals(ht_prepared)
    ht_final = calculate_daily_dose(ht_with_intervals)
    
    df = ht_final.to_pandas()
    df = df.rename(columns={'date_struct.year': 'year', 
                        'date_struct.month': 'month', 
                        'date_struct.day': 'day'})
    
    df.dropna(subset=['year', 'month', 'day'], inplace=True)
    df['date'] = pd.to_datetime(df[['year', 'month', 'day']])
    df = df.drop(['year', 'month', 'day'], axis=1)

    df['interval'] = df['interval'].astype('Int64')
    df['interval'] = df['interval'].replace(['', None], np.nan)
    df['daily_dose'] = df['daily_dose'].replace(['', None], np.nan)
    df['interval'] = df['interval'].fillna(df['quantity.value'])
    df['daily_dose'] = df['daily_dose'].fillna(df['doses.value'])
    df['daily_dose'] = df['daily_dose'].astype('Int64')

    df_grouped = df.sort_values(by=['eid', 'substance', 'date']).reset_index(drop=True)

    df_grouped = df_grouped.groupby(['eid', 'substance']).agg(
            date=('date', list),
            quantity_value=('quantity.value', list),
            quantity_is_days=('quantity.is_days', list),
            doses_unit=('doses.unit', list),
            doses_value=('doses.value', list),
            interval=('interval', list),
            daily_dose=('daily_dose', list)
        ).reset_index()

    df_grouped['all_data_collected'] = df_grouped.apply(
            lambda row: [
                {'date': row['date'][i],
                 'quantity_value': row['quantity_value'][i],
                 'quantity_is_days': row['quantity_is_days'][i],
                 'doses_unit': row['doses_unit'][i],
                 'doses_value': row['doses_value'][i],
                 'interval': row['interval'][i],
                 'daily_dose': row['daily_dose'][i]}
                for i in range(len(row['date']))
            ], axis=1
        )

    df_combined = df_grouped.drop(columns=['date', 'quantity_value', 'quantity_is_days', 'doses_unit', 'doses_value', 'interval', 'daily_dose'])

    df_combined['final_records'] = df_combined['all_data_collected'].apply(merge_consecutive_records)

    df_exploded = df_combined.drop(columns=['all_data_collected']).explode('final_records')

    df_final = pd.concat(
        [
            df_exploded.drop('final_records', axis=1),
            df_exploded['final_records'].apply(pd.Series)
        ],
        axis=1
    )

    df_final['sum_quantity'] = df_final['quantity_value'].apply(sum)
    filepath = f'../data/test_{drug_name}.csv'
    df_final.to_csv(filepath)

In [ ]:
def process_single_group(group_data_and_params):
    (eid, substance), group_df, sd_f, window = group_data_and_params
    
    group_df = group_df.sort_values(by=['date']).reset_index(drop=True)
    if 'interval' not in group_df.columns:
        group_df['interval'] = group_df['date'].diff().dt.days.fillna(0)

    therapies_df = process_therapies_with_dose_changes(group_df, substance, eid, sd_f=sd_f, sliding_window=window)

    if not therapies_df.empty:
        therapies_df['sd_factor'] = sd_f
        therapies_df['sliding_window'] = window
        return therapies_df
    else:
        return None

In [ ]:
import warnings

warnings.filterwarnings('ignore', category=RuntimeWarning)

In [ ]:
 sd_factors = [x / 100.0 for x in range(50, 300, 10)]
sliding_windows = [5, 7, 9, 11]
num_of_points = len(sd_factors) * len(sliding_windows)

for drug_name in drug_list[:1]:
    filepath = f'../data/test_{drug_name}.csv'
    df = pd.read_csv(filepath)
    
    groups = list(df.groupby(['eid', 'substance']))
    
    all_mse_results = []
    
    param_combinations = itertools.product(sd_factors, sliding_windows)
    
    progress_bar = tqdm(param_combinations, total=num_of_points, desc=f'Processing {drug_name}')
    
    for sd_f, window in progress_bar:
        args = [((eid, substance), group_df, sd_f, window) for (eid, substance), group_df in groups]

        with mp.Pool(mp.cpu_count()) as pool:
            results = pool.map(process_single_group, args)

        filtered_results = [res for res in results if res is not None]

        if filtered_results:
            final_results_df_test = pd.concat(filtered_results, ignore_index=True)
            all_mse_results.append(final_results_df_test)

    if all_mse_results:
        combined_results_df = pd.concat(all_mse_results, ignore_index=True)
        results_filepath = f'../data/results_{drug_name}.csv'
        combined_results_df.to_csv(results_filepath)
        print(f'Results for {drug_name} saved.')
    else:
        print(f'No results were generated for {drug_name}.')
    
    print('-' * 30)

In [ ]:
for drug_name in drug_list:
    results_filepath = f'../data/results_{drug_name}.csv'
    combined_results_df = pd.read_csv(results_filepath)
    
    print(f'Plots for substance: {drug_name}')

    person_means = combined_results_df.groupby(['eid', 'sd_factor', 'sliding_window']).agg(
        mean_nSMAE=('nSMAE', 'mean'),
        mean_nMSE=('nMSE', 'mean'),
        mean_duration=('therapy_duration_days', 'mean')
    ).reset_index()

    final_grouped_data = person_means.groupby(['sd_factor', 'sliding_window']).agg(
        final_mean_nSMAE=('mean_nSMAE', 'mean'),
        final_mean_nMSE=('mean_nMSE', 'mean'),
        final_mean_duration=('mean_duration', 'mean')
    ).reset_index()

    heatmap_nsmae_data = final_grouped_data.pivot(index='sd_factor', columns='sliding_window', values='final_mean_nSMAE')
    heatmap_nmse_data = final_grouped_data.pivot(index='sd_factor', columns='sliding_window', values='final_mean_nMSE')
    heatmap_duration_data = final_grouped_data.pivot(index='sd_factor', columns='sliding_window', values='final_mean_duration')

    plt.style.use('seaborn-v0_8-whitegrid')
    fig, axes = plt.subplots(1, 3, figsize=(22, 6))

    
    
    heat_data = heatmap_nsmae_data 
    sns.heatmap(heatmap_nsmae_data, ax=axes[0], annot=True, fmt=".3e", cmap="viridis")
    axes[0].set_title('The relationship between mean nSMAE and therapy parameters', fontsize=14)
    axes[0].set_xlabel('Sliding Window', fontsize=12)
    axes[0].set_ylabel('SD Factor', fontsize=12)
    
    min_nsmae = heatmap_nsmae_data.min()
    max_nsmae = heatmap_nsmae_data.max()
    normalized_nsmae = (heatmap_nsmae_data - min_nsmae) / (max_nsmae - min_nsmae)

    min_duration = heatmap_duration_data.min()
    max_duration = heatmap_duration_data.max()
    normalized_duration = (heatmap_duration_data - min_duration) / (max_duration - min_duration)
    
    epsilon = 1e-8
    heat_data = normalized_nsmae / (normalized_duration + epsilon)
    heat_data_log = np.log10(heat_data + epsilon)
    
    sns.heatmap(heat_data_log, ax=axes[1], annot=True, fmt=".2f", cmap="plasma")
    axes[1].set_title('mean nSMAE / average therapy duration (normalized, log scale)', fontsize=14)
    axes[1].set_xlabel('Sliding Window', fontsize=12)
    axes[1].set_ylabel('SD Factor', fontsize=12)

    sns.heatmap(heatmap_duration_data, ax=axes[2], annot=True, fmt=".1f", cmap="cividis")
    axes[2].set_title('The relationship between mean average therapy duration and therapy parameters', fontsize=14)
    axes[2].set_xlabel('Sliding Window', fontsize=12)
    axes[2].set_ylabel('SD Factor', fontsize=12)

    plt.tight_layout()
    plt.show()
    
    person_summary = combined_results_df.groupby('eid').agg(
        avg_duration=('therapy_duration_days', 'mean'),
        avg_nSMAE=('nSMAE', 'mean'),
        avg_nMSE=('nMSE', 'mean')
    ).reset_index()

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    sns.regplot(data=person_summary, x='avg_nSMAE', y='avg_duration', ax=axes[0], line_kws={"color": "red"})
    axes[0].set_title('average therapy duration vs. average nSMAE (per person)', fontsize=14)
    axes[0].set_xlabel('average nSMAE', fontsize=12)
    axes[0].set_ylabel('average therapy duration (days)', fontsize=12)

    sns.regplot(data=person_summary, x='avg_nMSE', y='avg_duration', ax=axes[1], line_kws={"color": "red"})
    axes[1].set_title('average therapy duration vs. average nMSE (per person)', fontsize=14)
    axes[1].set_xlabel('average nSMAE', fontsize=12)
    axes[1].set_ylabel('average therapy duration (days)', fontsize=12)

    plt.tight_layout()
    plt.show()

### Analysis of results and selection of parameters

In [ ]:
results = []

for drug in drug_list:
    path = f'data/final_result_{drug}.csv'
    df = pd.read_csv(path)
    df_sorted = df.sort_values(by='Mean Duration', ascending=False)
    length_of_top_30_percent_with_longest_duration = int(len(df_sorted) * 0.3)
    top_30_percent_df = df_sorted.head(length_of_top_30_percent_with_longest_duration).reset_index(drop=True)
    top_30_percent_df = top_30_percent_df.sort_values(by='nSMAE', ascending=True).reset_index(drop=True)
    row_with_min_error = top_30_percent_df.loc[0].copy()
    row_with_min_error['Drug'] = drug
    row_with_min_error = row_with_min_error[['SD Factor', 'Sliding Window', 'nSMAE', 'Mean Duration', 'Drug']]
    results.append(row_with_min_error)

results_df = pd.concat(results, axis=1).T.reset_index(drop=True)

In [ ]:
results_df

#### Choosen parameters:

SD_factor = 2.0

sliding_window = 5.0